# Section 2 — RAG over electropi.ai (LangChain + Chroma + Gemini)

A retrieval-augmented assistant whose knowledge base is **electropi.ai itself** —
the company's website and blog (the spec allows "your own domain docs — state your
choice"; this is the stated choice: real documents with citations the reviewer can
verify in seconds).

This notebook is **the** entry point: every stage of the pipeline runs as its own
cell, importing the `services/` modules (scraper, ingest, retriever, answerer,
cache, judge, pipeline) — the logic lives in modules, the notebook drives and
inspects it. Ships **pre-executed** with real outputs.

**Stack**: LangChain · Chroma (cosine) · `gemini-embedding-2` (768-dim MRL,
asymmetric task types) · `gemini-3.5-flash-lite` (answering + reranking) ·
`gemini-3.5-flash` (judge) · BM25 (`rank_bm25`)

**Pipeline**
```
question
  → semantic cache (cosine ≥ 0.92) ──hit──→ stored answer + citations (~0.5s)
  → dense gate (best cosine < 0.68 → refuse)          [threshold measured below]
  → multi-query expansion (2 Gemini rewrites)
  → hybrid retrieval: Chroma dense + BM25, per variant
  → Reciprocal Rank Fusion (k=60)
  → Gemini listwise rerank top-10 (all < 4/10 → refuse)
  → parent-section expansion (small-to-big)
  → Gemini answer, server-cached system prompt → [n] citations
```

**Reproduce**: `python3 -m venv .venv && .venv/bin/pip install -r requirements.txt`,
put `GOOGLE_API_KEY=...` in `.env`, then run all cells (~4 min, a few cents of
Gemini calls). The scrape step reuses the committed snapshot unless deleted.

In [1]:
import time, json, warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 120); pd.set_option("display.max_colwidth", 60)

import config

print("Answer/rerank model:", config.ANSWER_MODEL, "| judge:", config.JUDGE_MODEL)
print("Embeddings:", config.EMBEDDING_MODEL, f"({config.EMBEDDING_DIM}-dim)")
print("Guardrail gates: dense", config.DENSE_GATE, "| rerank", config.RERANK_GATE, "/10")

Answer/rerank model: gemini-3.5-flash-lite | judge: gemini-3.5-flash
Embeddings: models/gemini-embedding-2 (768-dim)
Guardrail gates: dense 0.68 | rerank 4 /10


## Step 1 — Scrape: structure-preserving, snapshot-first

Fixed page list (12 known pages — a BFS crawler would be ceremony for a site this
size), politeness delay, content-hash dedup. The part that matters for everything
downstream: extraction **keeps the heading structure** — headings become chunk
metadata and citation anchors.

Two things this site taught us (both found by checking the live DOM, both would
have silently gutted the knowledge base):
- `<main>` is a nearly-empty wrapper here; the real content sections are its
  *siblings* — extraction roots at `<body>` after stripping nav/header/footer.
- Case-study titles and result metrics ("95% prediction accuracy", "3.2× ROI")
  live in styled `<span>`s, not `<p>` tags, and card sub-headings
  ("Challenge"/"Solution") must be glued into their card's section or every case
  study shatters into anonymous fragments no query can match.

In [2]:
from services.scraper import scrape

if config.SCRAPE_FILE.exists():
    print("Using committed snapshot (delete data/scraped.json to re-scrape)\n")
    from utils.files import load_json
    pages = load_json(config.SCRAPE_FILE)
else:
    pages = scrape()

pd.DataFrame([{"page": p["url"].replace("https://electropi.ai", "") or "/",
               "category": p["category"], "sections": len(p["sections"]),
               "chars": sum(len(s["text"]) for s in p["sections"])} for p in pages])

Using committed snapshot (delete data/scraped.json to re-scrape)



,page,category,sections,chars
0,/,company,26,4120
1,/about,company,12,1964
2,/services,services,12,2034
3,/solutions,services,8,1506
4,/case-studies,case-studies,6,1625
5,/outsourcing,services,9,906
6,/ecosystem,company,6,1021
7,/contact,company,2,213
8,/blog/ocr-technology-enterprise-automation,blog,11,10163
9,/blog/ai-chatbots-for-enterprise,blog,24,8695


## Step 2 — Regex chunking (document-type aware)

Regex-driven chunking whose patterns are chosen for this exact corpus — a
marketing site plus long-form blog articles (one splitter never fits all
document types):

- **Structural cut**: heading-bounded sections from the scraper → each section is
  a **parent**; children are split inside it.
- **Sentence-boundary regex** `(?<=[.!?])\s+` (lookbehind, `is_separator_regex=True`)
  — a bare `"."` separator would cut inside "3.2", "99.9%", "et al.".
- **Junk regexes**: CTA lines, copyright, "N min read" — boilerplate repeated
  on every page carries no information worth indexing.
- **Contextual header**: every chunk is prefixed `page title | section heading`,
  so the embedding, BM25, and the LLM all see where the text lives.
- **Dedup** by exact text — marketing sites repeat blurbs across pages.

In [3]:
from services.ingest import build_chunks
from utils.text import CHUNK_SEPARATORS

print("separator regexes:", CHUNK_SEPARATORS, "\n")
chunks, parents = build_chunks(pages)

sample = next(c for c in chunks if c["metadata"]["heading"].startswith("Built by engineers"))
print("\n--- one chunk, text ---\n" + sample["text"][:400])
print("\n--- its metadata (citations are built from this) ---")
sample["metadata"]

separator regexes: ['\\n\\n', '(?<=[.!?])\\s+', '\\n', ' ', ''] 

145 chunks from 159 sections (0 duplicates dropped)

--- one chunk, text ---
Our Story & Mission | About ElectroPi | Built by engineers, for engineers.
ElectroPi was founded in 2016 by a team of AI researchers and software engineers who saw the gap between cutting-edge AI research and practical business applications.
What started as a small consulting practice has grown into a comprehensive AI engineering company, serving over 130 enterprise clients across industries.
Toda

--- its metadata (citations are built from this) ---


{'url': 'https://electropi.ai/about',
 'title': 'Our Story & Mission | About ElectroPi',
 'category': 'company',
 'heading': 'Built by engineers, for engineers.',
 'parent_id': 'about#1'}

## Step 3 — Embeddings + vector store

`gemini-embedding-2` with **asymmetric task types** — documents embedded as
`RETRIEVAL_DOCUMENT`, queries as `RETRIEVAL_QUERY` (Gemini optimizes the space
for exactly this pairing; skipping it leaves quality on the table). 768-dim MRL
truncation: 4× smaller index at near-full quality. Stored in Chroma, cosine
space, persisted to `index/`.

In [4]:
from services.ingest import ingest
ingest()   # full rebuild — deterministic, ~145 vectors, well under a minute

145 chunks from 159 sections (0 duplicates dropped)


Chroma index built: 145 vectors (768-dim, cosine) -> /Users/aa/Projects/Electro PI assesment/section2_rag/index/chroma


## Step 4 — Retrieval anatomy: dense · expansion · BM25 · RRF

One question, every intermediate ranking made visible. Dense finds meaning, BM25
finds exact words, multi-query expansion covers phrasings, and Reciprocal Rank
Fusion (k=60) merges all ranked lists without score calibration.

In [5]:
from services.pipeline import Pipeline

config.SEMANTIC_CACHE_FILE.unlink(missing_ok=True)   # demo from a cold cache
pipe = Pipeline()
R = pipe.retriever

q = "Why is Arabic voice AI particularly hard to build?"
vec = R.embedder.embed_query(q)

dense = R.dense_search(vec)[:5]
variants = R.expand(q)
bm25 = R.bm25_search(q)[:5]
fused = R.rrf([[cid for cid, _ in R.dense_search(vec)], R.bm25_search(q)])[:5]

print("query variants generated:")
for v in variants: print("  -", v)

h = lambda cid: R.chunks[cid]["metadata"]["heading"][:45]
pd.DataFrame({
    "dense (cosine)": [f"{h(c)}  ({s:.3f})" for c, s in dense],
    "bm25": [h(c) for c in bm25] + [""] * (5 - len(bm25[:5])),
    "RRF fused": [h(c) for c in fused],
})

query variants generated:
  - Why is Arabic voice AI particularly hard to build?
  - What makes developing Arabic speech recognition and voice assistants so difficult?
  - Challenges in creating voice AI technology for the Arabic language


,dense (cosine),bm25,RRF fused
0,Why Arabic voice AI is uniquely hard (0.850),Let's build your custom solution,Why Arabic voice AI is uniquely hard
1,Why Arabic Voice AI Matters for Customer Serv (0.811),Introduction,Introduction
2,Arabic OCR: The MENA-Specific Challenge (0.777),Why Arabic voice AI is uniquely hard,Why Arabic Voice AI Matters for Customer Serv
3,Why Egypt and Saudi Arabia Are Investing in A (0.759),Voice AI Agent vs Traditional IVR,Let's build your custom solution
4,Introduction (0.757),Conclusion,Arabic OCR: The MENA-Specific Challenge


## Step 5 — Listwise reranking (RankGPT-style)

One `gemini-3.5-flash-lite` call scores all 10 fused candidates 0–10 against the
question. Cheaper failure modes than pairwise, no extra infra like a
cross-encoder — and doubles as guardrail stage 2 (below).

In [6]:
candidates = R.rrf([[cid for cid, _ in R.dense_search(vec)], R.bm25_search(q)])[:config.RERANK_CANDIDATES]
reranked = R.rerank(q, candidates)
pd.DataFrame([{"relevance /10": rel, "chunk": h(cid),
               "page": R.chunks[cid]["metadata"]["url"].split("/")[-1] or "home"}
              for cid, rel in reranked])

,relevance /10,chunk,page
0,10,Why Arabic voice AI is uniquely hard,voice-ai-solutions
1,10,Why Arabic Voice AI Matters for Customer Serv,voice-ai-agent-for-customer-service
2,5,Introduction,voice-ai-solutions
3,5,Why Egypt and Saudi Arabia Are Investing in A,voice-ai-agent-for-customer-service
4,2,Common Mistakes to Avoid,ai-chatbots-for-enterprise
5,0,Let's build your custom solution,solutions
6,0,Arabic OCR: The MENA-Specific Challenge,ocr-technology-enterprise-automation
7,0,Voice AI Agent vs Traditional IVR,voice-ai-agent-for-customer-service
8,0,Conclusion,ocr-technology-enterprise-automation
9,0,What is Voice AI?,voice-ai-solutions


## Step 6 — The guardrail, caught in the act

Two measured stages. Watch both fire:

1. **Off-topic** ("capital of France") — dies at the *dense gate*: best cosine
   below the measured threshold, zero LLM calls spent.
2. **Near-topic trap** ("Who is the CEO?") — sounds like a company question,
   *passes* the dense gate… but no passage actually contains the answer, so the
   reranker scores everything 0/10 and the *rerank gate* refuses. This is the
   case a single-threshold guardrail gets wrong.

In [7]:
for trap in ["What is the capital of France?", "Who is the CEO of ElectroPi?"]:
    out = pipe.ask(trap)
    d = out["diagnostics"]
    print(f"Q: {trap}")
    print(f"   best dense cosine = {d['best_cosine']}  (gate {config.DENSE_GATE})")
    if "rerank" in d:
        print("   rerank scores:", [x["relevance"] for x in d["rerank"]])
    print(f"   -> {out['answer']}")
    print(f"   refused by: {d.get('gate', '—')}\n")

Q: What is the capital of France?
   best dense cosine = 0.549  (gate 0.68)
   -> I don't have relevant information about that in the ElectroPi knowledge base.
   refused by: dense gate (0.549 < 0.68)



Q: Who is the CEO of ElectroPi?
   best dense cosine = 0.778  (gate 0.68)
   rerank scores: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
   -> I don't have relevant information about that in the ElectroPi knowledge base.
   refused by: rerank gate (best 0 < 4)



## Step 7 — Guardrail thresholds: measured, not guessed

6 on-topic probes vs 6 adversarially-near off-topic probes (AI-flavored questions
the site does **not** cover). The dense gate (0.68) sits in the empty band
between the two distributions. Saved to `results/threshold_tuning.json`.

In [8]:
from services.pipeline import run_tune
run_tune(pipe)

  0.788  on-topic   What services does ElectroPi offer?


  0.819  on-topic   Tell me about ElectroPi's OCR work


  0.854  on-topic   Does ElectroPi build custom machine learning models?


  0.838  on-topic   What is ElectroPi's mission?


  0.857  on-topic   How do AI chatbots help enterprises?


...

  0.533  off-topic  How do I cook pasta carbonara?


  0.553  off-topic  Who won the 2022 World Cup?


  0.626  off-topic  How does a transformer neural network work mathematically?


  0.651  off-topic  What is the best GPU for training LLMs?


  0.564  off-topic  What's the weather in Cairo today?

on-topic  min=0.725  |  off-topic max=0.651  |  configured gate=0.68
Saved -> /Users/aa/Projects/Electro PI assesment/section2_rag/results/threshold_tuning.json


## Step 8 — The assessment examples (spec deliverable)

The spec asks for **3 example questions with the actual answers the pipeline
produced, plus a no-relevant-context case**. Here are five — 4 answerable + 1
unanswerable — through the **full pipeline**
(semantic cache cleared first: an evaluation run must not answer from cache).

A finding worth admitting: the *designed* unanswerable question was pricing —
then retrieval found real price ranges in a blog FAQ, so it stays as an answered
example (the system knew the corpus better than its author). The replacement
trap ("Who is the CEO?") is verified absent from the whole corpus.

In [9]:
from services.pipeline import fresh_examples, print_result
results = fresh_examples(pipe)
for r in results:
    print_result(r)


Q: When was ElectroPi founded, and how many enterprise clients does it serve?
A: ElectroPi was founded in 2016 and serves over 130 enterprise clients [1].

   [1] Our Story & Mission | About ElectroPi | Built by engineers, for engineers. — https://electropi.ai/about
   (3.48s · explicit-cache · 1138 prompt tokens read from server cache)

Q: Why is Arabic voice AI particularly hard to build?
A: Building Arabic voice AI is uniquely difficult for several reasons:

* **Dialect mixing and regional variations:** Systems must handle Egyptian Arabic, Saudi dialects (such as Najdi and Hejazi), Gulf and Levantine dialects, and Modern Standard Arabic (MSA)—often mixing dialects or MSA in the same call [1][2].
* **Code-switching:** Callers frequently switch between Arabic and English mid-sentence [1][2].
* **Noisy environments:** Callers speak from mobile networks, cars, cafes, and streets, which causes off-the-shelf ASR (Automatic Speech Recognition) to degrade quickly [1].
* **Named entities:**

## Step 9 — Instant answers: greetings + semantic cache

Two layers before any expensive work happens:

1. **Greeting patterns** — "hi", "thanks", "السلام عليكم"… are the most frequent
   inputs a public assistant sees. Exact-phrase table → canned reply, **zero API
   calls**. Real questions can never match (exact match only).
2. **Semantic cache** — every answered question is stored keyed by its query
   embedding. A *paraphrase* of an answered question returns the stored answer,
   with its citations, for the cost of one embedding call (the same embedding
   retrieval needs anyway — a lookup adds zero extra calls). The cache
   accumulates the real FAQ traffic automatically, and re-ingesting the index
   invalidates it.

In [10]:
g = pipe.ask("hi")
print(f"greeting: {g['seconds']}s ->", g["answer"][:90], "\n")

hit = pipe.ask("When did ElectroPi start and how many clients do they have?")
print(f"paraphrase: {hit['seconds']}s  (full pipeline was ~3-5s)")
print("matched:", hit["diagnostics"].get("matched_question"))
print("similarity:", hit["diagnostics"].get("similarity"))
print("\n" + hit["answer"])

greeting: 0.0s -> Hello! I'm the ElectroPi knowledge assistant. Ask me anything about ElectroPi — services,  



paraphrase: 0.31s  (full pipeline was ~3-5s)
matched: When was ElectroPi founded, and how many enterprise clients does it serve?
similarity: 0.949

ElectroPi was founded in 2016 and serves over 130 enterprise clients [1].


## Step 10 — Explicit prompt caching (server-side)

The system prompt — grounding rules + a **site map generated from the actual
scraped snapshot** (real grounding info: the model can see what the KB does and
does not cover) — is stored once on Gemini's servers and referenced by name on
every call. The cache name + expiry persist in `index/prompt_cache.json`, so it
survives across runs. Gemini's explicit-cache minimum is 1,024 tokens (measured);
below it the code falls back to sending the prompt inline and says so.

In [11]:
from utils.files import load_json
pc = load_json(config.PROMPT_CACHE_FILE)
print("mode:", pipe.answerer.mode)
print("server cache:", pc["name"])
print("tokens cached:", pc["cached_tokens"])
answered = [r for r in results if not r["refused"]]
print("tokens read from cache on each example answer:",
      [r["cached_tokens"] for r in answered])

mode: explicit-cache
server cache: cachedContents/nyb2w3236x9hwsvhjakcazbk7iudqzve89727lb8
tokens cached: 1138
tokens read from cache on each example answer: [1138, 1138, 1138, 1138]


## Step 11 — LLM judge (RAGAS-style rubric)

`gemini-3.5-flash` scores each answered example on **faithfulness** (every claim
supported by the sources?), **answer relevance**, and **context precision**
(1–5), and judges whether the refusal was correct. Same rubric-judge method as my
Section 3 blind evaluation — the standard approach in model eval and RLHF
preference pipelines. The judge sees only sources/question/answer, never the
pipeline. Artifacts → `results/`.

In [12]:
from services.judge import judge_examples
from services.pipeline import save_example_artifacts

judgments = judge_examples(results)
save_example_artifacts(results, judgments)

rows = []
for r, j in zip(results, judgments):
    if r["refused"]:
        rows.append({"question": r["question"][:50], "outcome": "REFUSED",
                     "judge": f"refusal_correct = {j['refusal_correct']}"})
    else:
        rows.append({"question": r["question"][:50], "outcome": "answered",
                     "judge": f"faith {j['faithfulness']}/5 · rel {j['answer_relevance']}/5 · prec {j['context_precision']}/5"})
pd.DataFrame(rows)

Saved qa_examples.md/.json + judge_results.json -> /Users/aa/Projects/Electro PI assesment/section2_rag/results


,question,outcome,judge
0,"When was ElectroPi founded, and how many enterpris",answered,faith 5/5 · rel 5/5 · prec 5/5
1,Why is Arabic voice AI particularly hard to build?,answered,faith 5/5 · rel 5/5 · prec 4/5
2,What results did ElectroPi's e-commerce case study,answered,faith 5/5 · rel 5/5 · prec 5/5
3,How much does ElectroPi charge for a chatbot proje,answered,faith 5/5 · rel 5/5 · prec 4/5
4,Who is the CEO of ElectroPi?,REFUSED,refusal_correct = True


## Playground — test any stage, ask anything

Three cells to poke the system directly:
1. `inspect(question)` — retrieval internals only (dense cosines, BM25, RRF,
   rerank scores), no answer generated. Use it to see *why* something is or
   isn't retrievable.
2. Ask a batch of questions end-to-end.
3. Ask one question.

In [13]:
def inspect(question: str):
    """Show every retrieval stage for a question — no answer generated."""
    v = R.embedder.embed_query(question)
    dense = R.dense_search(v)
    print(f"dense gate: best cosine {dense[0][1]:.3f} vs {config.DENSE_GATE}",
          "-> PASS" if dense[0][1] >= config.DENSE_GATE else "-> would REFUSE")
    variants = R.expand(question)
    print("variants:", variants[1:])
    rankings = [[cid for cid, _ in dense], R.bm25_search(question)]
    for var in variants[1:]:
        rankings.append([d.id for d in R.store.similarity_search(var, k=config.DENSE_TOP_K)])
        rankings.append(R.bm25_search(var))
    cands = R.rrf(rankings)[:config.RERANK_CANDIDATES]
    reranked = R.rerank(question, cands)
    return pd.DataFrame([{
        "rerank /10": rel,
        "dense cos": next((round(s2, 3) for c2, s2 in dense if c2 == cid), "—"),
        "chunk": R.chunks[cid]["metadata"]["heading"][:55],
        "page": R.chunks[cid]["metadata"]["url"].split("/")[-1] or "home",
    } for cid, rel in reranked])

inspect("What technologies does ElectroPi use for OCR?")

dense gate: best cosine 0.833 vs 0.68 -> PASS


variants: ['What optical character recognition software and tools are utilized by ElectroPi?', 'ElectroPi OCR technology stack and text extraction methods']


,rerank /10,dense cos,chunk,page
0,5,0.833,What Is OCR?,ocr-technology-enterprise-automation
1,5,0.789,Frequently Asked Questions,ocr-technology-enterprise-automation
2,5,0.806,How OCR Software Works,ocr-technology-enterprise-automation
3,5,0.805,The Future: From OCR to Full AI Document Intelligence,ocr-technology-enterprise-automation
4,5,0.8,Leading OCR Software and Cloud OCR Platforms,ocr-technology-enterprise-automation
5,5,0.799,Conclusion,ocr-technology-enterprise-automation
6,0,—,Enterprise Use Cases,ocr-technology-enterprise-automation
7,0,0.798,AI Engineering Services — OCR & Document Intelligence,home
8,0,0.782,Introduction,ocr-technology-enterprise-automation
9,0,0.79,Full-Spectrum AI Services — OCR & Document Intelligence,services


In [14]:
batch = [
    "What industries does ElectroPi have case studies in?",
    "What is Alcamp?",
    "hello",
]
for question in batch:
    print_result(pipe.ask(question))


Q: What industries does ElectroPi have case studies in?
A: ElectroPi has case studies in the following industries:
* Automotive Manufacturer [1]
* Leading Investment Bank [3]
* Healthcare Network [4]

   [1] AI Case Studies & Success Stories | ElectroPi | Industrial AI & Process Optimization — https://electropi.ai/case-studies
   [3] AI Case Studies & Success Stories | ElectroPi | OCR & Document Intelligence Platform — https://electropi.ai/case-studies
   [4] AI Case Studies & Success Stories | ElectroPi | Voice AI Patient Support System — https://electropi.ai/case-studies
   (3.32s · explicit-cache · 1138 prompt tokens read from server cache)



Q: What is Alcamp?
A: Alcamp is ElectroPi's dedicated educational arm and platform, providing comprehensive AI training programs, certifications, and academic support services for students and professionals to bridge the gap between academic knowledge and industry-ready AI expertise [1][2][4].

   [1] Alcamp by ElectroPi | AI Education & Training | Alcamp AI Education — https://electropi.ai/ecosystem
   [2] Alcamp by ElectroPi | AI Education & Training | Building the next generation of AI talent — https://electropi.ai/ecosystem
   [4] AI Engineering Services for Enterprise | ElectroPi | The ElectroPi Ecosystem — Alcamp — https://electropi.ai/
   (3.21s · explicit-cache · 1138 prompt tokens read from server cache)

Q: hello
A: Hello! I'm the ElectroPi knowledge assistant. Ask me anything about ElectroPi — services, case studies, the team, or its AI engineering blog (OCR, chatbots, Arabic voice AI, call-center AI).

   (0.0s · semantic-cache hit (=1.0 vs '(greeting pattern)'))


In [15]:
print_result(pipe.ask("Does ElectroPi work with clients outside Egypt?"))


Q: Does ElectroPi work with clients outside Egypt?
A: Yes, ElectroPi serves over 130 enterprise clients across industries worldwide [1][2].

   [1] AI Engineering Services for Enterprise | ElectroPi | Built for Enterprise Scale — https://electropi.ai/
   [2] Our Story & Mission | About ElectroPi | Built by engineers, for engineers. — https://electropi.ai/about
   (3.08s · explicit-cache · 1138 prompt tokens read from server cache)


## Summary

| Property | Result |
|---|---|
| Corpus | 12 pages / ~160 sections / ~145 chunks, snapshot committed |
| Guardrail separation | on-topic 0.725–0.857 vs off-topic 0.533–0.651 → gate 0.68 |
| Near-topic trap | passes dense gate, refused by rerank gate (0/10 across the board) |
| Example Q&As | 4 answered with verifiable citations, 1 correctly refused |
| Judge | 5/5/5 · 5/5/5 · 5/5/5 · 5/5/4 · refusal_correct = true |
| Semantic-cache hit | ~0.5–0.8 s vs ~3–5 s full pipeline |
| Prompt cache | ~1.1k system-prompt tokens read server-side per answer |

**Honest caveats** (full discussion in [NOTES.md](NOTES.md)): 5 examples and 12
tuning probes demonstrate behavior, not statistics — everything is reproducible
from this notebook. Judge and answerer share a model family (same-family judges
skew generous; mitigated by rubric + unsupported-claims listing). Re-ingesting
changed content must invalidate the semantic cache (evaluation runs here clear it).

**Production path**: these `services/` drop behind a FastAPI `routes/rag.py`,
Chroma → managed vector DB, semantic cache → Redis. Design write-up:
[NOTES.md](NOTES.md) · run instructions: [README.md](README.md).